In [5]:
import os
from pathlib import Path
testing = True

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))  # go to ~/crossdem/ by jumping up twice
OUT_DIR = os.path.join(BASE_DIR, "datasets")
if testing:
    OUT_DIR = os.path.join(BASE_DIR, "notebooks/out")
os.makedirs(OUT_DIR, exist_ok=True)

#with open(os.path.join(OUT_DIR, "output.txt"), "w") as f:
#    f.write("hello world")
#
#print(f"Written to {OUT_DIR}/output.txt")

## Download video
Given a radio radicale page url, extract the video embedding url and downloads it. Also embed metadata in the mp3 file.

In [24]:
import re
import json
import requests
import yt_dlp

METADATA_FIELDS = ("title", "description", "upload_date", "duration", "tags", "categories")

def get_audio_url(page_url: str) -> str:
    resp = requests.get(page_url, headers={"User-Agent": "Mozilla/5.0"})
    resp.raise_for_status()
    match = re.search(r'rtsp://[^"]+\.mp3', resp.text)
    if not match:
        raise ValueError("No rtsp audio link found on the page")
    return match.group(0)

def download_audio(url: str, out_dir: str) -> dict:
    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": f"{out_dir}/{url[-5:]}_%(title).10s.%(ext)s",        #~/notebooks/out/12345_abitoftitle.mp3
        "postprocessors": [
            {
                "key": "FFmpegExtractAudio",
                "preferredcodec": "mp3",
                "preferredquality": "192",
            },
            {
                "key": "FFmpegMetadata",
                "add_metadata": True,
            },
        ],
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        return ydl.sanitize_info(info)

def print_metadata(info: dict):
    print("\n--- Metadata ---")
    for field in METADATA_FIELDS:
        value = info.get(field)
        if value is not None:
            print(f"{field}: {value}")
    print("----------------\n")

if __name__ == "__main__":
    PAGE_URL = "https://www.radioradicale.it/scheda/157501/la-crisi-del-governo-fanfani-e-i-referendum-radicali-su-nucleare-e-responsabilita?i=1385823"
    #OUT_DIR = "./"

    try:
        info = download_audio(PAGE_URL, OUT_DIR)
    except yt_dlp.utils.DownloadError:
        print("yt-dlp couldn't extract directly, falling back to scrape...")
        audio_url = get_audio_url(PAGE_URL)
        print("Found:", audio_url)
        info = download_audio(audio_url, OUT_DIR)

    print_metadata(info)

[RadioRadicale] Extracting URL: https://www.radioradicale.it/scheda/157501/la-crisi-del-governo-fanfani-e-i-referendum-radicali-s...onsabilita?i=1385823
[RadioRadicale] 157501: Downloading webpage
[RadioRadicale] 157501-0: Downloading m3u8 information
[info] 157501: Downloading 1 format(s): 63
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 422
[download] Destination: /home/mhetac/Documents/GitHub/crossdem/notebooks/out/85823_＂La crisi .m4a
[download] 100% of   38.68MiB in 00:00:46 at 848.48KiB/s                 
[FixupM3u8] Fixing MPEG-TS in MP4 container of "/home/mhetac/Documents/GitHub/crossdem/notebooks/out/85823_＂La crisi .m4a"
[ExtractAudio] Destination: /home/mhetac/Documents/GitHub/crossdem/notebooks/out/85823_＂La crisi .mp3
Deleting original file /home/mhetac/Documents/GitHub/crossdem/notebooks/out/85823_＂La crisi .m4a (pass -k to keep)
[Metadata] Adding metadata to "/home/mhetac/Documents/GitHub/crossdem/notebooks/out/85823_＂La crisi .mp3"

--- Metadata --

All video metadata all extracted and embedded in the mp3 file 

In [25]:
for field, value in info.items():
    print(f"{field:<20}: {value}")

id                  : 157501
title               : "La crisi del governo Fanfani e i referendum radicali su nucleare e responsabilità civile dei magistrati..." servizio realizzato con documentazione tratta dall'archivio di Radio Radicale
formats             : [{'format_id': '63', 'format_index': None, 'url': 'https://video.radioradicale.it/store-78/_definst_/mp3:mp004/MP266557.mp3/chunklist_w86837949.m3u8', 'manifest_url': 'https://video.radioradicale.it/store-78/_definst_/mp3:mp004/MP266557.mp3/playlist.m3u8', 'tbr': 63.699, 'ext': 'm4a', 'fps': None, 'protocol': 'm3u8_native', 'preference': None, 'quality': None, 'has_drm': False, 'vcodec': 'none', 'acodec': 'mp4a.40.34', 'dynamic_range': None, 'audio_ext': 'm4a', 'video_ext': 'none', 'vbr': 0, 'abr': 63.699, 'resolution': 'audio only', 'aspect_ratio': None, 'http_headers': {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36', 'Accept': 'text/html,application

## Extract timestamps
Extract timestamps of audio fragment of specific politician
A video is something like [00....pannella......26......fanfani.....35.....someoneelse]

Most videos have explicit timestamps. 
```html
<div class="view-content int_menu">
      <ul>  <li class="int1576351 intervento d900">  
          
    
          <div class="int_name_section">    
          <h2>Marco Pannella</h2>    
          </div>    
          <div class="int_subtext">    
          <strong>stralcio di Stampa e Regime del 22 novembre 1999</strong>    
          </div>    
          <div class="durata"> 0:00 Durata: 15 min</div>    
              <div class="actions" style="display: none;">
						  <a href="/soggetti/86/marco-pannella" title="altri interventi dell'oratore"><i class="fa fa-external-link" aria-hidden="true"></i>altri interventi</a> 
					<a class="condividi condividi-intervento" href="#" data-param-i="1576351"><i class="fa fa-share-alt"></i>condividi intervento</a>
    </div>
    
</li>
  <li class="int1576350 intervento d1200">  
          
    
          <div class="int_name_section">    
          <h2>Massimo Bordin</h2><p>giornalista</p>    
          </div>    
          <div class="int_subtext">    
          <strong>L'on. Fanfani a una conferenza stampa del 22 marzo 1962</strong>    
          </div>    
          <div class="durata"> 0:15 Durata: 20 min</div>    
              <div class="actions" style="display: none;">
						  <a href="/soggetti/15014/massimo-bordin" title="altri interventi dell'oratore"><i class="fa fa-external-link" aria-hidden="true"></i>altri interventi</a> 
					<a class="condividi condividi-intervento" href="#" data-param-i="1576350"><i class="fa fa-share-alt"></i>condividi intervento</a>
    </div>
    
</li>
  <li class="int1576349 intervento d2700 selected">  
          
    
          <div class="int_name_section">    
          <h2>Amintore Fanfani</h2><p>DC</p>    
          </div>    
          <div class="int_subtext">    
          </div>    
          <div class="durata"> 0:35 Durata: 45 min</div>    
              <div class="actions" style="display: none;">
						  <a href="/soggetti/545/amintore-fanfani" title="altri interventi dell'oratore"><i class="fa fa-external-link" aria-hidden="true"></i>altri interventi</a> 
					<a class="condividi condividi-intervento" href="#" data-param-i="1576349"><i class="fa fa-share-alt"></i>condividi intervento</a>
    </div>
    
</li>
</ul>    </div>
```

## Cut audio 
Preserve only the sub-fragments of the specific politician

## Extract speech date
Uses BeautifulSup to parse all XML data and try to find out when a speech was made

In [26]:
import re
from datetime import date

import requests
from bs4 import BeautifulSoup


# ── HTTP ────────────────────────────────────────────────────────────────────

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64; rv:125.0) Gecko/20100101 Firefox/125.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "it-IT,it;q=0.9,en;q=0.5",
}


def fetch_page(url: str, timeout: int = 20) -> BeautifulSoup:
    resp = requests.get(url, headers=HEADERS, timeout=timeout)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


# ── Date patterns ───────────────────────────────────────────────────────────

MONTHS_IT = {
    "gennaio": 1, "febbraio": 2, "marzo": 3, "aprile": 4,
    "maggio": 5, "giugno": 6, "luglio": 7, "agosto": 8,
    "settembre": 9, "ottobre": 10, "novembre": 11, "dicembre": 12,
}
MONTHS_IT_SHORT = {
    "gen": 1, "feb": 2, "mar": 3, "apr": 4, "mag": 5, "giu": 6,
    "lug": 7, "ago": 8, "set": 9, "ott": 10, "nov": 11, "dic": 12,
}

RE_DATE_LONG   = re.compile(r"\b(\d{1,2})\s+(" + "|".join(MONTHS_IT) + r")\s+(\d{4})\b", re.I)
RE_DATE_SHORT  = re.compile(r"\b(\d{1,2})\s+(" + "|".join(MONTHS_IT_SHORT) + r")\s+(\d{4})\b", re.I)
RE_DATE_DOTTED = re.compile(r"\b(\d{1,2})\.(\d{2})\.(\d{4})\b")
RE_DATE_ISO    = re.compile(r"(\d{4})-(\d{2})-(\d{2})")


def _try_long(text):   m = RE_DATE_LONG.search(text);   return date(int(m[3]), MONTHS_IT[m[2].lower()],       int(m[1])) if m else None
def _try_short(text):  m = RE_DATE_SHORT.search(text);  return date(int(m[3]), MONTHS_IT_SHORT[m[2].lower()], int(m[1])) if m else None
def _try_dotted(text): m = RE_DATE_DOTTED.search(text); return date(int(m[3]), int(m[2]),                      int(m[1])) if m else None
def _try_iso(text):    m = RE_DATE_ISO.search(text);    return date(int(m[1]), int(m[2]),                      int(m[3])) if m else None

def find_date(text: str) -> date | None:
    return _try_long(text) or _try_short(text) or _try_dotted(text) or _try_iso(text)


# ── Extraction ──────────────────────────────────────────────────────────────

def extract_page_date(soup: BeautifulSoup) -> tuple[date | None, str]:
    """Page-level event date (meta > title > stamp > sommario prose)."""
    for attr, val in [("name", "dcterms.date"), ("property", "article:published_time")]:
        tag = soup.find("meta", {attr: val})
        if tag and tag.get("content"):
            d = _try_iso(tag["content"])
            if d: return d, "meta_tag"

    title = soup.find("title")
    if title:
        d = _try_dotted(title.get_text())
        if d: return d, "page_title"

    page_text = soup.get_text(" ", strip=True)
    d = _try_short(page_text)
    if d: return d, "page_stamp"

    d = _try_long(page_text)
    if d: return d, "page_sommario"

    return None, "not_found"


def extract_speaker_date(soup: BeautifulSoup, speaker: str) -> tuple[date | None, str]:
    """Per-speaker date from the Interventi block <li> containing an <h2>."""
    for li in (li for li in soup.find_all("li") if li.find("h2")):
        if speaker.lower() not in li.find("h2").get_text().lower():
            continue
        d = find_date(li.get_text(" ", strip=True))
        return (d, "speaker_block") if d else (None, "speaker_block_no_date")
    return None, "speaker_not_found"


def extract(url: str, speaker: str = "Fanfani", timeout: int = 20) -> dict:
    """
    Fetch a Radio Radicale scheda page and return a dict with:
      speech_date     – ISO string of the speaker's actual speech, or None
      date_source     – 'speaker_block' | 'page_event (...)' | 'llm_needed'
      page_event_date – ISO string of the page-level event date
      speaker_found   – bool
      error           – error string or None
    """
    result = dict(url=url, speaker_query=speaker, speaker_found=False,
                  speech_date=None, date_source=None, page_event_date=None, error=None)
    try:
        soup = fetch_page(url, timeout=timeout)
    except Exception as e:
        result["error"] = str(e)
        return result

    page_date, page_src = extract_page_date(soup)
    if page_date:
        result["page_event_date"] = page_date.isoformat()

    spk_date, spk_src = extract_speaker_date(soup, speaker)

    if spk_src == "speaker_not_found":
        result["error"] = f"'{speaker}' not found in interventi"
        result["date_source"] = "llm_needed"
        return result

    result["speaker_found"] = True

    if spk_date:
        result["speech_date"] = spk_date.isoformat()
        result["date_source"] = "speaker_block"
    elif page_date:
        result["speech_date"] = page_date.isoformat()
        result["date_source"] = f"page_event ({page_src})"
    else:
        result["date_source"] = "llm_needed"

    return result

In [30]:
# ── Run ─────────────────────────────────────────────────────────────────────

urls = [
    "https://www.radioradicale.it/scheda/157501/la-crisi-del-governo-fanfani-e-i-referendum-radicali-su-nucleare-e-responsabilita?i=1385823",
    "https://www.radioradicale.it/scheda/66298/i-congresso-nazionale-del-partito-popolare-italiano?i=2359758",
]
urls = [
    "https://www.radioradicale.it/scheda/157501/la-crisi-del-governo-fanfani-e-i-referendum-radicali-su-nucleare-e-responsabilita?i=1385823",
    "https://www.radioradicale.it/scheda/3913/crisi-di-governo-le-consultazioni-di-fanfani?i=2809940",
    "https://www.radioradicale.it/scheda/115568/le-elezioni-anticipate-della-primavera-87-il-governo-fanfani-lopposizione-radicale?i=1806617"
]

for url in urls:
    r = extract(url, speaker="Fanfani")
    for key, value in r.items():
        print(f"{key:<15} {value}")

url             https://www.radioradicale.it/scheda/157501/la-crisi-del-governo-fanfani-e-i-referendum-radicali-su-nucleare-e-responsabilita?i=1385823
speaker_query   Fanfani
speaker_found   True
speech_date     1987-04-21
date_source     speaker_block
page_event_date 2004-09-03
error           None
url             https://www.radioradicale.it/scheda/3913/crisi-di-governo-le-consultazioni-di-fanfani?i=2809940
speaker_query   Fanfani
speaker_found   True
speech_date     1982-11-27
date_source     page_event (meta_tag)
page_event_date 1982-11-27
error           None
url             https://www.radioradicale.it/scheda/115568/le-elezioni-anticipate-della-primavera-87-il-governo-fanfani-lopposizione-radicale?i=1806617
speaker_query   Fanfani
speaker_found   True
speech_date     1987-04-20
date_source     speaker_block
page_event_date 1999-11-24
error           None
